# Speculative Decoding Experiment

**Phases 1-6**: Data preparation -> Baseline -> AR Grid A (0.5B) -> AR Grid B (1.5B) -> Quality/Error Analysis -> Synthesis/Visualization

| Config | Draft | k | Regime |
|--------|-------|---|--------|
| Baseline x 2 | - | - | det / stoch |
| Grid A x 6 | Qwen2.5-0.5B | {4,8,16} | det / stoch |
| Grid B x 6 | Qwen2.5-1.5B | {4,8,16} | det / stoch |

**Total runs:** 2 baselines + 12 speculative = 14 configurations

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is on the path (robust for local and Colab execution)
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Core imports
import torch
import pandas as pd

from config import (
    TARGET_MODEL_ID, DRAFT_MODELS, DATASETS, REGIMES,
    DRAFT_LENGTHS, RESULTS_DIR, STABILITY_DIR, FIGURES_DIR,
    SEED, STABILITY_SEEDS, MANIFESTS_DIR,
)
from utils import set_seed, get_env_info

# Print environment
env = get_env_info()
for k, v in env.items():
    print(f"{k}: {v}")

## Phase 1 — Data Preparation & Reproducibility Lock

Load GSM8K (300), MMLU (5×100), CNN/DailyMail (200) with `seed=42`, freeze manifests.

In [ ]:
from data_loader import load_all_datasets, freeze_manifests, save_full_data, load_from_manifests

# Check if manifests already exist (skip download if so)
manifests_exist = all(
    (MANIFESTS_DIR / f"{t}_data.json").exists()
    for t in ["gsm8k", "mmlu", "cnndm"]
)

if manifests_exist:
    print("Manifests found — loading from disk (no re-download)")
    data = load_from_manifests()
else:
    print("Downloading and sampling datasets…")
    data = load_all_datasets()
    freeze_manifests(data)
    save_full_data(data)

# Quick sanity check
for task, samples in data.items():
    print(f"  {task}: {len(samples)} samples, first id: {samples[0]['sample_id']}")

### Verify Tokenizer Compatibility

In [ ]:
from data_loader import verify_tokenizer_compatibility

compatible = verify_tokenizer_compatibility()
assert compatible, "Tokenizer mismatch — cannot proceed with speculative decoding!"

## Phase 2 — Baseline Benchmarking

Run Qwen2.5-7B-Instruct autoregressive decoding on all 1,000 samples in both regimes.

In [ ]:
from baseline import load_target_model, run_baseline

# Load model once, reuse for both regimes
target_model, target_tokenizer = load_target_model()

In [ ]:
# --- Deterministic baseline ---
print("=" * 60)
print("Baseline: DETERMINISTIC regime")
print("=" * 60)
baseline_det = run_baseline(data, "deterministic", target_model, target_tokenizer)

In [ ]:
# --- Stochastic baseline ---
print("=" * 60)
print("Baseline: STOCHASTIC regime")
print("=" * 60)
baseline_stoch = run_baseline(data, "stochastic", target_model, target_tokenizer)

In [ ]:
# --- Evaluate baseline quality ---
from evaluate import evaluate_results

print("\nBaseline quality — Deterministic:")
base_quality_det = evaluate_results(baseline_det, data)

print("\nBaseline quality — Stochastic:")
base_quality_stoch = evaluate_results(baseline_stoch, data)

In [ ]:
# --- Quick latency summary ---
from metrics import compute_latency_metrics

print("\nBaseline latency — Deterministic:")
base_lat_det = compute_latency_metrics(baseline_det)
for k, v in base_lat_det.items():
    print(f"  {k}: {v}")

print("\nBaseline latency — Stochastic:")
base_lat_stoch = compute_latency_metrics(baseline_stoch)
for k, v in base_lat_stoch.items():
    print(f"  {k}: {v}")

## Phase 3 — Speculative Grid A: Qwen2.5-0.5B Draft

Run speculative decoding with 0.5B draft across k ∈ {4, 8, 16} × {deterministic, stochastic} = 6 configs.

In [ ]:
from speculative import load_draft_model, run_speculative_grid

# Load 0.5B draft model once
draft_05_model, draft_05_tokenizer = load_draft_model("0.5B")

In [ ]:
spec_results_05 = {}

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"0.5B_k{k_val}_{regime_name}"
        print(f"\n{'=' * 60}")
        print(f"Speculative: draft=0.5B, k={k_val}, regime={regime_name}")
        print(f"{'=' * 60}")
        
        results = run_speculative_grid(
            data, "0.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_05_model, draft_05_tokenizer,
        )
        spec_results_05[config_key] = results

print(f"\n✓ Grid A complete: {len(spec_results_05)} configs")

In [ ]:
# --- Grid A: Evaluate quality and compute metrics ---
from metrics import compute_acceptance_metrics, compute_speedup, compute_quality_delta

grid_a_summary = []
for config_key, results in spec_results_05.items():
    parts = config_key.split("_")  # e.g. '0.5B_k4_deterministic'
    k_val = int(parts[1][1:])
    regime = parts[2]
    
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch
    
    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)
    
    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")
    
    grid_a_summary.append({
        "config": config_key, "draft": "0.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_a = pd.DataFrame(grid_a_summary)
df_grid_a

## Phase 4 — Speculative Grid B: Qwen2.5-1.5B Draft

Run speculative decoding with 1.5B draft across the same 6 configs, plus stability analysis.

In [ ]:
# Free 0.5B draft to save memory, then load 1.5B
del draft_05_model, draft_05_tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

draft_15_model, draft_15_tokenizer = load_draft_model("1.5B")

In [ ]:
spec_results_15 = {}

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"1.5B_k{k_val}_{regime_name}"
        print(f"\n{'=' * 60}")
        print(f"Speculative: draft=1.5B, k={k_val}, regime={regime_name}")
        print(f"{'=' * 60}")
        
        results = run_speculative_grid(
            data, "1.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_15_model, draft_15_tokenizer,
        )
        spec_results_15[config_key] = results

print(f"\n✓ Grid B complete: {len(spec_results_15)} configs")

In [ ]:
# --- Grid B: Evaluate quality and compute metrics ---
grid_b_summary = []
for config_key, results in spec_results_15.items():
    parts = config_key.split("_")
    k_val = int(parts[1][1:])
    regime = parts[2]
    
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch
    
    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)
    
    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")
    
    grid_b_summary.append({
        "config": config_key, "draft": "1.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_b = pd.DataFrame(grid_b_summary)
df_grid_b

In [ ]:
# --- Combine all 12 configs into master table ---
df_all = pd.concat([df_grid_a, df_grid_b], ignore_index=True)
print("\nFull 12-config results matrix:")
display(df_all[["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"]])

# Save master table
df_all.to_csv(RESULTS_DIR / "all_configs_summary.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_summary.csv'}")

### Phase 4b — Stability Analysis (Top 2 Configs)

Identify the top-2 configs by speedup (subject to |ΔQ| ≤ 1.0) and re-run with seeds {42, 123, 999}.

In [ ]:
from metrics import compute_speedup_stability
from speculative import run_stability_analysis

# Filter configs meeting quality constraint |ΔQ| <= 1.0 for ALL tasks
delta_cols = [c for c in df_all.columns if c.startswith("delta_")]
if delta_cols:
    df_qualified = df_all[df_all[delta_cols].abs().max(axis=1) <= 1.0].copy()
else:
    # If delta columns use dQ_ prefix
    dq_cols = [c for c in df_all.columns if c.startswith("dQ_")]
    df_qualified = df_all[df_all[dq_cols].abs().max(axis=1) <= 1.0].copy()

# Select top 2 by speedup
top2 = df_qualified.nlargest(2, "S")
print("Top 2 configs (|ΔQ| ≤ 1.0):")
display(top2[["config", "S", "alpha"]])

In [ ]:
# Run stability analysis for top 2 configs
stability_results = {}

for _, row in top2.iterrows():
    draft_label = row["draft"]
    k_val = int(row["k"])
    regime = row["regime"]
    config_key = row["config"]
    
    print(f"\n{'=' * 60}")
    print(f"Stability analysis: {config_key}")
    print(f"{'=' * 60}")
    
    # Load correct draft model
    if draft_label == "0.5B":
        dm, dt = load_draft_model("0.5B")
    else:
        dm, dt = draft_15_model, draft_15_tokenizer
    
    seed_runs = run_stability_analysis(
        data, draft_label, k_val, regime,
        target_model, target_tokenizer, dm, dt,
    )
    
    # Compute speedup per seed
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    speedups = []
    for sr in seed_runs:
        s = compute_speedup(baseline_ref, sr["results"])
        speedups.append(s)
        print(f"  seed={sr['seed']}: S={s:.4f}")
    
    sigma = compute_speedup_stability(speedups)
    print(f"  σ_S = {sigma:.4f}")
    stability_results[config_key] = {"speedups": speedups, "sigma_S": sigma}

print("\n✓ Stability analysis complete")

## Phase 5 — Quality and Error Analysis

Systematic analysis of disagreement patterns between speculative and baseline outputs, including length buckets, taxonomy, and rejection behavior proxies.

In [ ]:
from difflib import SequenceMatcher
import numpy as np
from evaluate import cnndm_rouge_l

# Merge all speculative runs
all_spec_results = {**spec_results_05, **spec_results_15}

# Baseline lookup by regime + sample_id
baseline_lookup = {
    "deterministic": {r["sample_id"]: r for r in baseline_det},
    "stochastic": {r["sample_id"]: r for r in baseline_stoch},
}

rows = []
for config_name, cfg_rows in all_spec_results.items():
    regime = cfg_rows[0]["regime"] if cfg_rows else "deterministic"
    for r in cfg_rows:
        sid = r["sample_id"]
        base = baseline_lookup[regime].get(sid)
        if base is None:
            continue

        pred_text = (r.get("output_text") or "").strip()
        base_text = (base.get("output_text") or "").strip()

        pred_words = len(pred_text.split())
        base_words = len(base_text.split())
        disagree = pred_text != base_text
        sim_ratio = SequenceMatcher(None, pred_text, base_text).ratio()

        if pred_words <= 16:
            length_bucket = "short"
        elif pred_words <= 64:
            length_bucket = "medium"
        else:
            length_bucket = "long"

        rouge_to_base = np.nan
        if r["task"] == "cnndm" and pred_text and base_text:
            rouge_to_base = cnndm_rouge_l(pred_text, base_text)

        rows.append({
            "config": config_name,
            "draft": r["draft"],
            "k": r["k"],
            "regime": regime,
            "task": r["task"],
            "sample_id": sid,
            "disagree": disagree,
            "pred_words": pred_words,
            "base_words": base_words,
            "length_bucket": length_bucket,
            "sim_ratio": sim_ratio,
            "rouge_to_base": rouge_to_base,
            "alpha_sample": (r.get("total_accepted", 0) / r.get("total_proposed", 1)) if r.get("total_proposed", 0) > 0 else np.nan,
        })

df_phase5 = pd.DataFrame(rows)
print(f"Constructed Phase 5 analysis table: {len(df_phase5)} rows")
display(df_phase5.head(10))

In [ ]:
# Disagreement rates by config/task and by output length bucket

df_disagree = df_phase5[df_phase5["disagree"]].copy()

disagree_rate = (
    df_phase5.groupby(["config", "task"], as_index=False)["disagree"]
    .mean()
    .rename(columns={"disagree": "disagreement_rate"})
)

bucket_dist = (
    df_disagree.groupby(["config", "length_bucket"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not bucket_dist.empty:
    bucket_dist["pct_within_config"] = (
        bucket_dist["count"]
        / bucket_dist.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

print("Disagreement rate by config/task:")
display(disagree_rate.sort_values(["disagreement_rate", "config"], ascending=[False, True]).head(20))

print("\nLength bucket distribution among disagreement cases:")
display(bucket_dist.sort_values(["config", "length_bucket"]))

In [ ]:
# Error taxonomy + rejection clustering proxy

def classify_error(row):
    if not row["disagree"]:
        return "match"

    pred_w = max(row["pred_words"], 1)
    base_w = max(row["base_words"], 1)

    # Truncation: output substantially shorter than baseline output.
    if pred_w < 0.6 * base_w:
        return "truncation"

    # Semantic drift: low lexical similarity and (for CNNDM) low ROUGE-L to baseline output.
    if row["sim_ratio"] < 0.25:
        return "semantic_drift"
    if row["task"] == "cnndm" and not np.isnan(row["rouge_to_base"]) and row["rouge_to_base"] < 0.2:
        return "semantic_drift"

    return "substitution"


df_phase5["error_type"] = df_phase5.apply(classify_error, axis=1)

taxonomy = (
    df_phase5[df_phase5["error_type"] != "match"]
    .groupby(["config", "error_type"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not taxonomy.empty:
    taxonomy["pct_within_config"] = (
        taxonomy["count"]
        / taxonomy.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

# Rejection-position clustering requires verify-step position logs.
# Current CSV rows do not keep verify_log, so we provide an acceptance-ratio proxy.
proxy = df_phase5.dropna(subset=["alpha_sample"]).copy()
proxy["rejection_proxy"] = pd.cut(
    proxy["alpha_sample"],
    bins=[-1, 0.33, 0.66, 1.0],
    labels=["early_reject_proxy", "mid_reject_proxy", "late_reject_proxy"],
)
rej_proxy_table = (
    proxy.groupby(["config", "rejection_proxy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not rej_proxy_table.empty:
    rej_proxy_table["pct_within_config"] = (
        rej_proxy_table["count"]
        / rej_proxy_table.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
print(f"verify_log persisted in run outputs: {has_verify_logs}")
if not has_verify_logs:
    print("NOTE: true token-position rejection clustering is not possible from current saved rows; using acceptance-ratio proxy instead.")

print("\nError taxonomy among disagreement cases:")
display(taxonomy.sort_values(["config", "count"], ascending=[True, False]))

print("\nRejection clustering proxy:")
display(rej_proxy_table.sort_values(["config", "rejection_proxy"]))

## Phase 6 — Synthesis and Visualization

Generate Pareto/acceptance figures, synthesis tables, and an execution-plan audit for missing items.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

df_plot = df_all.copy()
delta_cols = [c for c in df_plot.columns if c.startswith("delta_")]
if not delta_cols:
    delta_cols = [c for c in df_plot.columns if c.startswith("dQ_")]

df_plot["max_abs_delta"] = df_plot[delta_cols].abs().max(axis=1) if delta_cols else 0.0

# Pareto scatter: Speedup vs max quality drift
plt.figure(figsize=(8, 5))
ax = sns.scatterplot(
    data=df_plot,
    x="S",
    y="max_abs_delta",
    hue="draft",
    style="regime",
    s=90,
)
ax.set_title("Pareto: Speedup vs Max |Delta Q|")
ax.set_xlabel("Speedup S (x)")
ax.set_ylabel("Max |Delta Q| (pp)")
plt.tight_layout()
pareto_path = FIGURES_DIR / "pareto_speedup_vs_quality_delta.png"
plt.savefig(pareto_path, dpi=160)
plt.show()

# Acceptance vs k grouped by draft and regime
alpha_k = (
    df_plot.groupby(["draft", "regime", "k"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.lineplot(data=alpha_k, x="k", y="alpha", hue="draft", style="regime", marker="o")
ax.set_title("Acceptance Rate alpha vs Draft Length k")
ax.set_xlabel("k")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_k_path = FIGURES_DIR / "acceptance_vs_k.png"
plt.savefig(alpha_k_path, dpi=160)
plt.show()

# Acceptance by regime (temperature proxy)
alpha_reg = (
    df_plot.groupby(["regime", "draft"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=alpha_reg, x="regime", y="alpha", hue="draft")
ax.set_title("Acceptance Rate alpha by Regime")
ax.set_xlabel("Regime")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_reg_path = FIGURES_DIR / "acceptance_by_regime.png"
plt.savefig(alpha_reg_path, dpi=160)
plt.show()

print("Saved figures:")
print(f"  - {pareto_path}")
print(f"  - {alpha_k_path}")
print(f"  - {alpha_reg_path}")

In [ ]:
# Synthesis tables for manuscript

results_matrix_cols = [
    "config", "draft", "k", "regime", "S", "alpha", "B_eff",
    "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms",
    "gsm8k", "mmlu", "cnndm",
]
results_matrix_cols = [c for c in results_matrix_cols if c in df_all.columns]

results_matrix = df_all[results_matrix_cols].sort_values(["draft", "k", "regime"]).copy()
display(results_matrix)
results_matrix.to_csv(RESULTS_DIR / "table_full_results_matrix.csv", index=False)

# Best speculative config per task + baseline comparison
quality_table_rows = []
baseline_quality_map = {
    "deterministic": base_quality_det,
    "stochastic": base_quality_stoch,
}

for task in ["gsm8k", "mmlu", "cnndm"]:
    if task not in df_all.columns:
        continue
    best_idx = df_all[task].idxmax()
    best_row = df_all.loc[best_idx]
    regime = best_row["regime"]
    baseline_score = baseline_quality_map[regime][task]
    quality_table_rows.append({
        "task": task,
        "baseline_regime": regime,
        "baseline_score": baseline_score,
        "best_spec_config": best_row["config"],
        "best_spec_score": best_row[task],
        "delta_spec_minus_base": round(best_row[task] - baseline_score, 2),
    })

df_quality_compare = pd.DataFrame(quality_table_rows)
print("Best speculative config per task (vs matching baseline regime):")
display(df_quality_compare)
df_quality_compare.to_csv(RESULTS_DIR / "table_quality_comparison.csv", index=False)

print("Saved tables:")
print(f"  - {RESULTS_DIR / 'table_full_results_matrix.csv'}")
print(f"  - {RESULTS_DIR / 'table_quality_comparison.csv'}")

In [ ]:
# Audit checklist: what is still missing for Phase 6 paper completion
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "sec").exists():
    # Fallback to notebook path root behavior if executed from elsewhere
    project_root = Path(SRC_DIR).parent

checks = []

# 1) Missing sections requested by execution plan
for rel in ["sec/01b_related_work.tex", "sec/06_results.tex"]:
    p = project_root / rel
    checks.append({
        "item": f"Section file exists: {rel}",
        "status": "PASS" if p.exists() else "MISSING",
        "evidence": str(p),
    })

# 2) Abstract placeholders still present?
abstract_path = project_root / "sec/00_abstract.tex"
if abstract_path.exists():
    abstract_text = abstract_path.read_text(encoding="utf-8", errors="ignore")
    placeholders_present = any(tok in abstract_text for tok in ["[X]", "[Y]", "[Z]"])
    checks.append({
        "item": "Abstract placeholders resolved ([X],[Y],[Z])",
        "status": "MISSING" if placeholders_present else "PASS",
        "evidence": "Placeholders found" if placeholders_present else "No placeholders found",
    })

# 3) Rejection-position instrumentation availability
has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
checks.append({
    "item": "Token-position rejection logs persisted",
    "status": "PASS" if has_verify_logs else "MISSING",
    "evidence": "verify_log present in result rows" if has_verify_logs else "verify_log dropped in run_speculative_grid output rows",
})

# 4) Visualization module implementation
viz_path = project_root / "src/visualize.py"
if viz_path.exists():
    viz_text = viz_path.read_text(encoding="utf-8", errors="ignore")
    stub = "implemented later" in viz_text.lower()
    checks.append({
        "item": "src/visualize.py implemented",
        "status": "MISSING" if stub else "PASS",
        "evidence": "Stub marker found" if stub else "No stub marker found",
    })

# 5) Ensure phase outputs written
required_outputs = [
    project_root / "results/all_configs_summary.csv",
    project_root / "results/all_configs_combined.csv",
    project_root / "results/table_full_results_matrix.csv",
    project_root / "results/table_quality_comparison.csv",
]
for out in required_outputs:
    checks.append({
        "item": f"Output generated: {out.name}",
        "status": "PASS" if out.exists() else "PENDING",
        "evidence": str(out),
    })

df_audit = pd.DataFrame(checks)
print("Phase 6 audit report:")
display(df_audit)

missing_items = df_audit[df_audit["status"].isin(["MISSING", "PENDING"])]
if missing_items.empty:
    print("\nAll checklist items passed.")
else:
    print("\nItems still pending/missing:")
    display(missing_items)

## Summary

Display the final results matrix with all 12 configs + stability info.

In [ ]:
# Final summary table
print("=" * 80)
print("EXPERIMENT COMPLETE — FULL RESULTS MATRIX")
print("=" * 80)

display_cols = [c for c in ["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"] if c in df_all.columns]

print("\n--- Latency & Throughput ---")
display(df_all[display_cols].sort_values("R_tok_mean", ascending=False))

quality_cols = [c for c in df_all.columns if c in ["gsm8k", "mmlu", "cnndm"]]
if quality_cols:
    print("\n--- Quality (%) ---")
    display(df_all[["config"] + quality_cols])

# Best config under quality constraint
if not df_qualified.empty:
    best = df_qualified.loc[df_qualified["S"].idxmax()]
    print(f"\n★ Best config (|ΔQ| ≤ 1.0): {best['config']}")
    print(f"  Speedup: {best['S']:.2f}x, α: {best['alpha']:.3f}")

# Save combined table
df_all.to_csv(RESULTS_DIR / "all_configs_combined.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_combined.csv'}")

# Stability
if stability_results:
    print("\n--- Stability (σ_S) ---")
    for cfg, info in stability_results.items():
        print(f"  {cfg}: σ_S = {info['sigma_S']:.4f}, seeds = {info['speedups']}")